# Практика 2. Работа с LLM через API (GigaChat + LangChain)


## 0. Импорт библиотек и подготовка окружения

Перед запуском установите зависимости командой `python -m pip install -r requirements.txt`.


In [25]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser

## 1. Импорт библиотек и подключение к GigaChat

In [26]:
load_dotenv()
API_KEY = os.getenv("GIGA_KEY")

if not API_KEY:
    raise ValueError("Переменная окружения GIGA_KEY не задана.")

llm = GigaChat(
    credentials=API_KEY,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

print(llm.invoke("Привет! Ответь коротко.").content)

Привет. Коротко.


## 2. Выбор варианта и загрузка данных



In [27]:
VARIANT = "20"  

csv_path = f"rental_{VARIANT}.csv"

try:
    df = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, sep=";", encoding="cp1251")

required_columns = {"text", "amount"}
if not required_columns.issubset(df.columns):
    raise ValueError(f"В CSV должны быть столбцы {required_columns}. Найдены: {list(df.columns)}")

if len(df) != 15:
    raise ValueError(f"Ожидается 15 заявок, а найдено {len(df)}.")

df["amount"] = pd.to_numeric(df["amount"], errors="raise").astype(int)
print(f"Загружено заявок: {len(df)}")
display(df.head())

Загружено заявок: 15


,amount,text
0,6,Здравствуйте! Ищем жильё для 2 взрослых и 4 де...
1,2,"Добрый вечер! Ищем жилье на 2 человек, с 3 сен..."
2,4,Ищем жильё на 4 взрослых с 31.08.18 по 07.08.1...
3,3,"Здравствуйте, интересует жилье с 17.08 по 25.0..."
4,3,Добрый день. Ищем комфортное жильё в Лазаревс...


## 3. Задание 1 — базовый промпт

In [28]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "Ты — эксперт по анализу текстовых заявок на аренду жилья. "
    "Определи общее количество людей, которые будут проживать."
)

user_prompt = HumanMessagePromptTemplate.from_template(
    "Заявка: {text}\n\n"
    "Извлеки общее количество проживающих. "
    "Верни только одно целое число, без пояснений."
)

chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])
chain = chat_prompt | llm | StrOutputParser()

print(chain.invoke({"text": "Семья из трех человек снимает квартиру"}))  # ожидается 3

3


## 4. Обработка 15 заявок


In [29]:
## Обработка 15 заявок из варианта 20

# Берём все 15 текстов заявок из файла rental_20.csv.
test_texts = df["text"].tolist()

results = []
for text in test_texts:
    try:
        result = chain.invoke({"text": text})
        result = str(result).strip()
        results.append(result)
        print(f"Заявка: {text}\nКоличество проживающих: {result}\n---")
    except Exception as error:
        results.append(f"ERROR: {error}")
        print(f"Ошибка обработки заявки: {error}")

# Сохраняем ответы модели в отдельный столбец.
df["result"] = results
display(df[["text", "amount", "result"]])

Заявка: Здравствуйте! Ищем жильё для 2 взрослых и 4 детей (8,5,4,2) в период с 29.05.2018 по 09.06.2018 года с кухней, недалёко от моря! Предложения в л/с
Количество проживающих: 6
---
Заявка: Добрый вечер! Ищем жилье на 2 человек, с 3 сентября на 7 дней.
Количество проживающих: 2
---
Заявка: Ищем жильё на 4 взрослых с 31.08.18 по 07.08.18, желательно центр
Количество проживающих: 4
---
Заявка: Здравствуйте, интересует жилье с 17.08 по 25.08 двое взрослых и девочка 14 лет.
Количество проживающих: 3
---
Заявка: Добрый день. Ищем  комфортное жильё в Лазаревском. 
2 взрослых и Ребенок 3 года 
С 15 по 22 августа.
Количество проживающих: 3
---
Заявка: Ищем жильё в Лазаревское заезд 13 июля на 10 дней, 2 взр, дети 2 года и 5 лет, предпочтения душ, туалет, холодильник, кондиционер в номере, парковка для авто,  рассмотрим варианты, не курим
Количество проживающих: 4
---
Заявка: Здравствуйте! Интересует жилье недорого 1-10 сентября.2 взр и 2 детей-1 номер нужен.В ЛС.
Количество проживающих: 4
-

    text                                                                   amount result
  0 Здравствуйте! Ищем жильё для 2 взрослых и 4 детей (8,5,4,2) в пе...         6      6
  1 Добрый вечер! Ищем жилье на 2 человек, с 3 сентября на 7 дней.              2      2
  2 Ищем жильё на 4 взрослых с 31.08.18 по 07.08.18, желательно центр           4      4
  3 Здравствуйте, интересует жилье с 17.08 по 25.08 двое взрослых и ...         3      3
  4 Добрый день. Ищем комфортное жильё в Лазаревском. 2 взрослых и Р...         3      3
  5 Ищем жильё в Лазаревское заезд 13 июля на 10 дней, 2 взр, дети 2...         4      4
  6 Здравствуйте! Интересует жилье недорого 1-10 сентября.2 взр и 2 ...         4      4
  7 Здравствуйте🤗 Ищем жильё в центре рядом с морем🌊 Цена за номер...         2      2
  8 Здравствуйте! Ищем жильё с 15 по 23 августа 2 взрослых 1 ребёнок...         3      3
  9 Сниму жилье в июне у моря 2 взрослых до 1500 в сутки                        2      2
 10 Добрый день! Интере